# Sesión 1 — ¿Qué es aprender de datos? Python y NumPy para ML
**Curso: Introducción a la Inteligencia Artificial · IEEE CIS USFQ**

**Objetivos de hoy**
1. Entender qué significa que un modelo *aprenda*: **modelo**, **parámetros** y **función de pérdida**.
2. Repasar lo esencial de Python.
3. Usar arreglos de NumPy y pensar en forma *vectorizada*.
4. Traducir notación matemática ($\sum$, vectores, matrices, $\hat{\mathbf y}=\mathbf X\mathbf w$) a código.

> Guarda una copia: **Archivo → Guardar una copia en Drive**.

**Cómo usar este notebook en clase:** tenlo abierto junto a las slides. Cada diapositiva azul *“En tu notebook”* te indica qué parte ejecutar, y el pie de cada slide muestra en qué parte vamos. Antes de ejecutar una celda, intenta predecir qué va a imprimir; después, cambia valores y vuelve a ejecutarla.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

def comprobar(prueba, mensaje):
    """Ejecuta una prueba y muestra ✅ o ❌ sin detener el notebook."""
    try:
        ok = bool(prueba())
    except Exception as e:
        ok = False
        mensaje += f"  (error: {type(e).__name__}: {e})"
    print(("✅ " if ok else "❌ ") + mensaje)
    return ok

RUTA = "/content/sample_data/california_housing_train.csv"
if not os.path.exists(RUTA):  # por si no estás en Colab
    RUTA = "https://download.mlcc.google.com/mledu-datasets/california_housing_train.csv"

: 

---
## Parte 1 · El problema que vamos a resolver en el curso
Tenemos datos de ~17 000 barrios de California. Para cada uno conocemos su **ingreso mediano** y el **valor mediano de sus viviendas**. Pregunta: *si te doy el ingreso de un barrio nuevo, ¿puedes predecir el valor de sus casas?*

In [ ]:
df = pd.read_csv(RUTA)
x = df["median_income"].to_numpy()                   # decenas de miles de USD
y = df["median_house_value"].to_numpy() / 100_000    # cientos de miles de USD

print("Número de barrios:", len(x))
plt.scatter(x, y, s=3, alpha=0.2)
plt.xlabel("Ingreso mediano $x$ (decenas de miles de USD)")
plt.ylabel("Valor mediano $y$ (cientos de miles de USD)")
plt.title("Datos de entrenamiento")
plt.show()

### Un modelo es una función con perillas
Proponemos el modelo más simple posible, una recta:
$$\hat y = w\,x + b$$
- $x$: la entrada (ingreso). $\hat y$ ("y gorro"): la **predicción**.
- $w$ (peso) y $b$ (sesgo) son los **parámetros**: las perillas que podemos girar.

**Aprender = encontrar los valores de $w$ y $b$ que hacen buenas predicciones.**

👉 Cambia `w` y `b` en la celda de abajo hasta que la recta se vea bien.

In [ ]:
def dibujar_recta(w, b):
    plt.scatter(x, y, s=3, alpha=0.15, label="datos")
    xs = np.linspace(0, 15, 100)
    plt.plot(xs, w * xs + b, color="C3", lw=2.5, label=f"$\\hat y = {w}x + {b}$")
    plt.ylim(0, 5.5)
    plt.xlabel("Ingreso $x$"); plt.ylabel("Valor $y$")
    plt.legend()
    plt.show()

dibujar_recta(w=0.3, b=1.0)   # ← prueba otros valores

### ¿Cuál recta es *mejor*? Necesitamos un número
"Se ve bien" no es una respuesta que una computadora pueda optimizar. Medimos el **error** de cada predicción, $y_i - \hat y_i$, lo elevamos al cuadrado (para que los errores positivos y negativos no se cancelen y para castigar más los errores grandes) y promediamos:
$$\text{MSE}(w,b) = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - \hat y_i\right)^2 = \frac{1}{n}\sum_{i=1}^{n}\left(y_i - (w x_i + b)\right)^2$$
Esta es la **función de pérdida** (*loss*): un solo número que dice qué tan mal lo hace el modelo. **Más bajo es mejor.** Al final de hoy sabrás programarla en una línea.

| Pieza | En nuestro problema |
|---|---|
| **Modelo** | $\hat y = wx + b$ |
| **Parámetros** | $w,\ b$ |
| **Pérdida** | MSE |
| **Optimización** | ¿cómo encontrar los $w,b$ que minimizan el MSE? → sesiones 2 y 3 |

---
## Parte 2 · Python exprés
Solo lo que vamos a usar. Ejecuta cada celda y **predice el resultado antes de ejecutarla**.
### Variables y tipos

In [ ]:
n_casas = 17000          # int
precio = 2.5             # float
ciudad = "Quito"         # str
es_caro = precio > 3     # bool
print(type(n_casas), type(precio), type(ciudad), type(es_caro))
print(f"En {ciudad} hay {n_casas} casas; ¿es caro? {es_caro}")   # f-string

### Listas, índices y rebanadas
Python empieza a contar en **0**. `lista[a:b]` toma desde `a` hasta `b-1`.

In [ ]:
precios = [1.2, 3.4, 2.2, 5.0, 0.9]
print(precios[0], precios[-1])     # primero y último
print(precios[1:3])                # elementos 1 y 2
print(len(precios))
precios.append(4.1)
print(precios)

### Ciclos, condicionales y comprensiones de lista

In [ ]:
total = 0
for p in precios:
    if p > 2:
        total += p
print("Suma de los precios mayores a 2:", total)

# Lo mismo en una línea con una comprensión de lista:
print(sum([p for p in precios if p > 2]))

cuadrados = [k**2 for k in range(5)]
print(cuadrados)

### Funciones

In [ ]:
def promedio(valores):
    """Devuelve el promedio de una lista de números."""
    return sum(valores) / len(valores)

print(promedio(precios))

### Diccionarios (los veremos en Pandas)

In [ ]:
casa = {"ingreso": 3.5, "valor": 2.1, "ciudad": "Fresno"}
print(casa["valor"])
for clave, valor in casa.items():
    print(clave, "→", valor)

---
## Parte 3 · NumPy: la base de todo
Las listas de Python son flexibles pero **lentas** para hacer cuentas con millones de números. NumPy ofrece el **arreglo** (`ndarray`): números del mismo tipo, guardados juntos en memoria y con operaciones escritas en C.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
print(a, a.dtype, a.shape)

print(np.zeros(3), np.ones(3))
print(np.arange(0, 10, 2))          # como range, pero devuelve arreglo
print(np.linspace(0, 1, 5))         # 5 puntos igualmente espaciados entre 0 y 1

rng = np.random.default_rng(42)     # generador aleatorio con semilla (reproducible)
print(rng.normal(0, 1, size=4))

### Operaciones elemento a elemento (vectorización)
Las operaciones aritméticas actúan **sobre cada elemento**, sin escribir ciclos.

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([10.0, 20.0, 30.0])
print(a + b)
print(a * b)        # ¡elemento a elemento! (no es el producto punto)
print(a ** 2)
print(2 * a + 1)    # el escalar se aplica a cada elemento (broadcasting)
print(np.sqrt(a), np.exp(a))

### ¿De verdad es más rápido?

In [ ]:
grande = rng.normal(size=1_000_000)

t0 = time.time()
s = 0.0
for v in grande:
    s += v**2
t_ciclo = time.time() - t0

t0 = time.time()
s2 = np.sum(grande**2)
t_numpy = time.time() - t0

print(f"Ciclo de Python: {t_ciclo:.3f} s")
print(f"NumPy:           {t_numpy:.4f} s   → unas {t_ciclo / max(t_numpy, 1e-6):.0f} veces más rápido")

### Indexación, máscaras booleanas y agregaciones

In [ ]:
v = np.array([3.0, -1.0, 4.0, -1.5, 5.0])
print(v[0], v[-1], v[1:4])
mascara = v > 0
print(mascara)
print(v[mascara])                   # solo los positivos
print(v.sum(), v.mean(), v.max(), v.argmax())

### Arreglos 2D (matrices) y el parámetro `axis`

In [ ]:
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(M.shape)            # (filas, columnas)
print(M[1, 2])            # fila 1, columna 2 → 6
print(M[:, 0])            # toda la columna 0
print(M.sum(axis=0))      # suma por columnas (colapsa las filas)
print(M.sum(axis=1))      # suma por filas
print(M.T)                # transpuesta

---
## Parte 4 · De la notación matemática al código
| Matemáticas | Se lee | NumPy |
|---|---|---|
| $x_i$ | el elemento $i$ del vector $\mathbf x$ | `x[i]` (¡ojo! Python empieza en 0) |
| $\sum_{i=1}^{n} x_i$ | suma de todos los $x_i$ | `np.sum(x)` |
| $\bar x = \frac{1}{n}\sum_i x_i$ | media | `np.mean(x)` |
| $\mathbf x \in \mathbb R^n$ | vector de $n$ números reales | arreglo con `shape == (n,)` |
| $\mathbf X \in \mathbb R^{n\times d}$ | matriz de $n$ filas y $d$ columnas | arreglo con `shape == (n, d)` |
| $\mathbf a\cdot\mathbf b = \mathbf a^{\mathsf T}\mathbf b = \sum_i a_i b_i$ | producto punto | `a @ b` o `np.dot(a, b)` |
| $\lVert \mathbf a\rVert = \sqrt{\sum_i a_i^2}$ | norma (longitud) | `np.linalg.norm(a)` |
| $\mathbf X\mathbf w$ | producto matriz–vector | `X @ w` |
| $\mathbf X^{\mathsf T}$ | transpuesta | `X.T` |

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])

print("Σ a_i      =", np.sum(a))
print("a · b      =", a @ b, "=", 1*4 + 2*5 + 3*6)
print("‖a‖        =", np.linalg.norm(a), "=", np.sqrt(1 + 4 + 9))

### El producto matriz–vector: predecir para *todos* los datos a la vez
Cada **fila** de $\mathbf X$ es un dato (un barrio) y cada **columna** una característica. Si el modelo es $\hat y = w_1 x_1 + w_2 x_2 + \dots + w_d x_d$, entonces la predicción del dato $i$ es el producto punto de su fila con $\mathbf w$, y **todas** las predicciones juntas son
$$\hat{\mathbf y} = \mathbf X \mathbf w, \qquad \hat y_i = \sum_{j=1}^{d} X_{ij}\, w_j .$$

In [ ]:
X = np.array([[3.0, 20.0],     # barrio 1: ingreso 3.0, antigüedad 20 años
              [5.0, 10.0],     # barrio 2
              [1.5, 45.0]])    # barrio 3
w = np.array([0.4, 0.01])

y_hat_ciclo = np.array([sum(X[i, j] * w[j] for j in range(2)) for i in range(3)])
y_hat = X @ w
print(X.shape, "@", w.shape, "→", y_hat.shape)
print(y_hat_ciclo)
print(y_hat)

**Truco del sesgo:** si añadimos una columna de unos a $\mathbf X$, el sesgo $b$ se vuelve un peso más. Así $\hat y = wx + b$ también se escribe como $\hat{\mathbf y} = \mathbf X\mathbf w$.

In [ ]:
x_mini = np.array([1.0, 2.0, 3.0])
X_mini = np.column_stack([x_mini, np.ones_like(x_mini)])   # columnas: [x, 1]
w_mini = np.array([0.5, 1.0])                              # [w, b]
print(X_mini)
print(X_mini @ w_mini, "=", 0.5 * x_mini + 1.0)

---
## Parte 5 · La función de pérdida en código
Con todo lo anterior, el MSE se escribe en **una línea** y sin ciclos.

In [ ]:
def mse(y, y_pred):
    return np.mean((y - y_pred) ** 2)

# Ejemplo pequeño (el mismo de la pizarra)
x_p = np.array([1.0, 2.0, 3.0])
y_p = np.array([2.0, 4.0, 5.0])
print("MSE de ŷ = 1.5x + 0.5:", mse(y_p, 1.5 * x_p + 0.5))

# Datos reales
for w_, b_ in [(0.3, 1.0), (0.4, 0.5), (0.5, 0.0)]:
    print(f"w={w_}, b={b_}  →  MSE = {mse(y, w_ * x + b_):.4f}")

### 🏆 Competencia: ¿quién encuentra la recta con menor MSE?
Prueba valores de `w` y `b`. Anota tu mejor resultado. Piensa: **¿cómo harías para que la computadora los busque sola?** Esa es la pregunta de la sesión 2.

In [ ]:
w_mio, b_mio = 0.4, 0.5      # ← cambia estos valores
print(f"MSE = {mse(y, w_mio * x + b_mio):.5f}")
dibujar_recta(w_mio, b_mio)

---
## Ejercicios
Completa cada celda reemplazando `pass` y ejecuta la celda de comprobación que le sigue. Están ordenados de menor a mayor dificultad.

**Ejercicio 1.** Escribe `media_lista(valores)` que calcule el promedio de una lista **usando un ciclo `for`** (sin `sum`, sin NumPy).

In [ ]:
def media_lista(valores):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
comprobar(lambda: abs(media_lista([1, 2, 3, 4]) - 2.5) < 1e-12, "media_lista([1,2,3,4]) == 2.5")
comprobar(lambda: abs(media_lista(list(y[:100])) - np.mean(y[:100])) < 1e-9, "coincide con np.mean")

**Ejercicio 2.** Usando una comprensión de lista, crea `impares_al_cuadrado` con los cuadrados de los números impares entre 1 y 19 (inclusive).

In [ ]:
impares_al_cuadrado = None   # TU CÓDIGO AQUÍ

In [ ]:
comprobar(lambda: impares_al_cuadrado == [1, 9, 25, 49, 81, 121, 169, 225, 289, 361], "lista correcta")

**Ejercicio 3.** Escribe `producto_punto(a, b)` **con un ciclo** (sin `@` ni `np.dot`), siguiendo la definición $\mathbf a\cdot\mathbf b = \sum_i a_i b_i$.

In [ ]:
def producto_punto(a, b):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
u = rng.normal(size=50); v_ = rng.normal(size=50)
comprobar(lambda: abs(producto_punto([1, 2, 3], [4, 5, 6]) - 32) < 1e-12, "[1,2,3]·[4,5,6] == 32")
comprobar(lambda: abs(producto_punto(u, v_) - u @ v_) < 1e-9, "coincide con u @ v")

**Ejercicio 4.** Del arreglo `y` (valores de vivienda), calcula **sin ciclos**:
- `n_caras`: cuántos barrios tienen valor mayor a 3 (300 000 USD).
- `media_baratas`: el valor promedio de los barrios con valor menor a 1.

*Pista:* `np.sum` de un arreglo booleano cuenta los `True`.

In [ ]:
n_caras = None        # TU CÓDIGO AQUÍ
media_baratas = None  # TU CÓDIGO AQUÍ

In [ ]:
comprobar(lambda: n_caras == len([v for v in y if v > 3]), "n_caras correcto")
comprobar(lambda: abs(media_baratas - np.mean([v for v in y if v < 1])) < 1e-9, "media_baratas correcta")

**Ejercicio 5.** Escribe `predecir(X, w)` que devuelva $\hat{\mathbf y}=\mathbf X\mathbf w$ y `mse_lineal(X, y, w)` que devuelva el MSE de esas predicciones. Ambas sin ciclos.

In [ ]:
def predecir(X, w):
    # TU CÓDIGO AQUÍ
    pass

def mse_lineal(X, y, w):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
X_t = np.column_stack([x, np.ones_like(x)])
comprobar(lambda: np.allclose(predecir(X_t, np.array([0.4, 0.5])), 0.4 * x + 0.5), "predecir correcto")
comprobar(lambda: abs(mse_lineal(X_t, y, np.array([0.4, 0.5])) - mse(y, 0.4 * x + 0.5)) < 1e-12, "mse_lineal correcto")

**Ejercicio 6 (reto).** Busca la mejor recta **por fuerza bruta**: prueba todas las combinaciones de `w` en `np.linspace(0, 1, 101)` y `b` en `np.linspace(-1, 2, 101)` y guarda en `mejor_w`, `mejor_b` y `mejor_mse` la combinación con menor MSE. ¿Cuántas veces evaluaste el MSE? ¿Qué pasaría con 10 parámetros en vez de 2?

In [ ]:
mejor_w, mejor_b, mejor_mse = None, None, None   # TU CÓDIGO AQUÍ

In [ ]:
comprobar(lambda: mejor_mse is not None and mejor_mse <= mse(y, 0.4 * x + 0.5) + 1e-12, "tu búsqueda mejora (o iguala) w=0.4, b=0.5")
def _minimo_rejilla():
    bs = np.linspace(-1, 2, 101)
    return min(np.mean((y[None, :] - (w_ * x[None, :] + bs[:, None]))**2, axis=1).min()
               for w_ in np.linspace(0, 1, 101))
comprobar(lambda: abs(mejor_mse - _minimo_rejilla()) < 1e-9, "encontraste el mínimo de la rejilla")

---
### Para la próxima sesión
- Termina los ejercicios y la **Guía de estudio 1** (PDF en Classroom).
- Opcional: 3Blue1Brown, *Essence of Linear Algebra*, capítulos 1–3.
- Pregunta para pensar: en el ejercicio 6 evaluaste el MSE más de 10 000 veces para ajustar solo 2 parámetros. ¿Existe una forma de saber **hacia dónde** mover $w$ y $b$ para que el error baje, sin probar a ciegas?